# 04: Product Categories & Seller Performance

Which product categories get the worst ratings, and which sellers consistently
underperform? Both questions need a minimum-review-count filter — without it, a
category or seller with 2 orders and one bad review would falsely rank as the
worst.

In [ ]:
import os
for folder in ["../data/raw", "../data/interim", "../data/processed",
                "../reports/figures", "../notebooks", "../src"]:
    os.makedirs(folder, exist_ok=True)
print("All folders confirmed.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 110

products = pd.read_csv("../data/interim/products_clean.csv")
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
reviews = pd.read_csv("../data/interim/reviews_clean.csv")

## Q3: Which product categories get the worst ratings?

Join items → products (for category) → reviews (for score). Filter to
categories with at least 30 reviews so small-sample noise doesn't produce a
misleading "worst" list.

In [ ]:
item_cat = items.merge(products[["product_id", "product_category_name_english"]], on="product_id", how="left")
item_cat = item_cat.merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
item_cat = item_cat.dropna(subset=["review_score"])

cat_scores = (
    item_cat.groupby("product_category_name_english")["review_score"]
    .agg(["mean", "count"])
    .query("count >= 30")
    .sort_values("mean")
)
print("WORST categories:\n", cat_scores.head(10))
print("\nBEST categories:\n", cat_scores.tail(10))

## Chart: 10 worst-rated categories

Horizontal bars because category names are long — vertical bars would need
rotated, hard-to-read labels.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
worst10 = cat_scores.head(10).sort_values("mean")
ax.barh(worst10.index, worst10["mean"], color="firebrick")
ax.set_xlabel("Average Review Score")
ax.set_title("10 Worst-Rated Product Categories (min. 30 reviews)")
ax.set_xlim(0, 5)
plt.savefig("../reports/figures/04_worst_categories.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
best10 = cat_scores.tail(10).sort_values("mean")
ax.barh(best10.index, best10["mean"], color="seagreen")
ax.set_xlabel("Average Review Score")
ax.set_title("10 Best-Rated Product Categories (min. 30 reviews)")
ax.set_xlim(0, 5)
plt.savefig("../reports/figures/05_best_categories.png", bbox_inches="tight")
plt.show()

## Q4: Which sellers consistently underperform?

Same logic as categories: minimum review count filter (20 here, since sellers
have fewer reviews on average than categories), then also look specifically at
which sellers rack up the most 1-star reviews in absolute terms; a seller with
a mediocre *average* but huge volume can generate more unhappy customers than
one with a terrible average but tiny volume.

In [ ]:
items_s = items.merge(reviews[["order_id", "review_score"]], on="order_id", how="left").dropna(subset=["review_score"])

seller_scores = (
    items_s.groupby("seller_id")["review_score"]
    .agg(["mean", "count"])
    .query("count >= 20")
    .sort_values("mean")
)
print("Worst-performing sellers (min 20 reviews):\n", seller_scores.head(15))

one_star = items_s[items_s["review_score"] == 1]
top_offenders = one_star["seller_id"].value_counts()
print("\nTop sellers by raw count of 1-star reviews:\n", top_offenders.head(10))

## Chart: distribution of seller performance

A histogram shows whether bad sellers are a small tail of outliers or a broad
spread — this changes the recommendation (audit a handful of sellers vs.
address a systemic issue).

In [ ]:
fig, ax = plt.subplots()
sns.histplot(seller_scores["mean"], bins=30, kde=True, ax=ax)
ax.axvline(seller_scores["mean"].mean(), color="red", linestyle="--", label="Overall average")
ax.set_xlabel("Seller's Average Review Score")
ax.set_title("Distribution of Seller Performance (min. 20 reviews)")
ax.legend()
plt.savefig("../reports/figures/08_seller_distribution.png", bbox_inches="tight")
plt.show()